<a href="https://colab.research.google.com/github/sakshi987123/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sakshi987123/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip install -q duckdb pandas pyarrow huggingface_hub


Load your Hugging Face token

In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded successfully!" if HF_TOKEN else "HF_TOKEN not found.")

Token loaded successfully!


Connect DuckDB

In [4]:
import duckdb

con = duckdb.connect()

print("DuckDB connected successfully!")

DuckDB connected successfully!


Load the HTTPFS extension

In [5]:
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

print("Extensions loaded!")

Extensions loaded!


Connect to the FlyRank warehouse

In [6]:
con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [7]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5;
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [8]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_query_90d']}
LIMIT 5;
""").df()

,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [9]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    ROUND(
        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks * 100.0 / gsc_impressions
            ELSE 0
        END,
        2
    ) AS ctr,
    gsc_sum_position
FROM {TABLES['fact_daily']}
WHERE gsc_data_available IS TRUE
LIMIT 10;
""").df()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_sum_position
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,30,0,0.0,115
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,5,0,0.0,358
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,1,0,0.0,34
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,6,0,0.0,140
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,5,0,0.0,89
5,2025-01-27,client_9958f0a7ae1df715,content_c782fa8abd4fce5e,21,0,0.0,1050
6,2025-01-27,client_9958f0a7ae1df715,content_ae5e5fd6edff550f,13,0,0.0,127
7,2025-01-27,client_9958f0a7ae1df715,content_a64143f6e4a21ffe,29,0,0.0,356
8,2025-01-27,client_9958f0a7ae1df715,content_e281674658070602,5,0,0.0,103
9,2025-01-27,client_9958f0a7ae1df715,content_658f53fa439c66ca,8,0,0.0,304


Decide your baseline rule Check Signal 1 (Impressions)

In [10]:
con.sql(f"""
SELECT
CASE
    WHEN gsc_impressions < 10 THEN '0-9'
    WHEN gsc_impressions < 100 THEN '10-99'
    WHEN gsc_impressions < 1000 THEN '100-999'
    ELSE '1000+'
END AS impression_bucket,

COUNT(*) AS n

FROM {TABLES['fact_daily']}

WHERE gsc_data_available IS TRUE

GROUP BY 1

ORDER BY 1;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_bucket,n
0,0-9,13359275
1,10-99,11753537
2,100-999,3669213
3,1000+,188026


Most content has low to moderate impressions, while a much smaller set receives over 1,000 impressions. High-impression pages are suitable candidates for optimization because even small CTR improvements can generate meaningful additional traffic.

Check Signal 2 (CTR)

In [11]:
con.sql(f"""
SELECT
CASE
    WHEN gsc_impressions = 0 THEN 'No Impressions'
    WHEN (gsc_clicks * 100.0 / gsc_impressions) < 1 THEN '<1%'
    WHEN (gsc_clicks * 100.0 / gsc_impressions) < 3 THEN '1-3%'
    WHEN (gsc_clicks * 100.0 / gsc_impressions) < 5 THEN '3-5%'
    ELSE '>5%'
END AS ctr_bucket,

COUNT(*) AS n

FROM {TABLES['fact_daily']}

WHERE gsc_data_available IS TRUE

GROUP BY 1

ORDER BY 1;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ctr_bucket,n
0,1-3%,1050296
1,3-5%,275419
2,<1%,27244437
3,>5%,399899


A large proportion of content has CTR below 1%, indicating many pages have opportunities for title or snippet improvements. This supports using CTR as a baseline signal for ranking optimization actions.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
ranked = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    ROUND(
        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks * 100.0 / gsc_impressions
            ELSE 0
        END,
        2
    ) AS ctr,

    CASE
        WHEN gsc_impressions >= 1000
             AND (gsc_clicks * 100.0 / NULLIF(gsc_impressions,0)) < 1
        THEN 100

        WHEN gsc_impressions >= 100
             AND (gsc_clicks * 100.0 / NULLIF(gsc_impressions,0)) < 3
        THEN 70

        ELSE 30
    END AS score,

    CASE
        WHEN gsc_impressions >= 100
             AND (gsc_clicks * 100.0 / NULLIF(gsc_impressions,0)) < 3
        THEN 'CTR_FIX'
        ELSE 'LOW_PRIORITY'
    END AS reason_code,

    CASE
        WHEN gsc_impressions >= 100
             AND (gsc_clicks * 100.0 / NULLIF(gsc_impressions,0)) < 3
        THEN 'Improve Title'
        ELSE 'Monitor'
    END AS action

FROM {TABLES['fact_daily']}
WHERE gsc_data_available IS TRUE
AND report_date BETWEEN '2026-03-01' AND '2026-03-31'

ORDER BY score DESC
LIMIT 100000
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

View the Top 10

Save csv

In [13]:
import os

os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("baseline_action_score.csv saved successfully!")

baseline_action_score.csv saved successfully!


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [14]:
top20 = ranked.head(20)

top20

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,score,reason_code,action
0,2026-03-01,client_73cda7b4e4f265ea,content_91798557b91aa5bd,1082,2,0.18,100,CTR_FIX,Improve Title
1,2026-03-01,client_73cda7b4e4f265ea,content_8eba2d99238592aa,2105,12,0.57,100,CTR_FIX,Improve Title
2,2026-03-01,client_73cda7b4e4f265ea,content_1380729256470720,1074,0,0.00,100,CTR_FIX,Improve Title
3,2026-03-01,client_73cda7b4e4f265ea,content_4bcb72eff0cc03f9,1224,4,0.33,100,CTR_FIX,Improve Title
4,2026-03-01,client_73cda7b4e4f265ea,content_c9f840183215651b,1692,0,0.00,100,CTR_FIX,Improve Title
5,2026-03-01,client_73cda7b4e4f265ea,content_ada72ae2a2b33800,1314,4,0.30,100,CTR_FIX,Improve Title
6,2026-03-01,client_73cda7b4e4f265ea,content_9b6b685c29cdde6b,1146,1,0.09,100,CTR_FIX,Improve Title
7,2026-03-01,client_73cda7b4e4f265ea,content_1f3195e2491d9314,2442,4,0.16,100,CTR_FIX,Improve Title
8,2026-03-01,client_73cda7b4e4f265ea,content_c684e826fd2cc874,1806,0,0.00,100,CTR_FIX,Improve Title
9,2026-03-01,client_73cda7b4e4f265ea,content_91d8af19d84c05a6,1154,0,0.00,100,CTR_FIX,Improve Title


## Top-20 Review

### Rank 1
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,082 impressions with a CTR of only 0.18%, indicating good visibility but poor click-through performance.
- **What would make it wrong:** Seasonal search behaviour or a recent title/meta description update not yet reflected in the data.

### Rank 2
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 2,105 impressions with a CTR of 0.57%, suggesting strong optimization potential.
- **What would make it wrong:** User intent may not match the page content, so changing the title may not improve CTR.

### Rank 3
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,074 impressions with 0 clicks (CTR 0.00%), making it a high-priority candidate.
- **What would make it wrong:** The page may have been recently published or recently updated.

### Rank 4
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,224 impressions with a CTR of 0.33%, showing high visibility but low engagement.
- **What would make it wrong:** The search query may not align with the page's content.

### Rank 5
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,692 impressions with 0 clicks (CTR 0.00%), indicating a strong optimization opportunity.
- **What would make it wrong:** The impressions may come from low-intent or irrelevant searches.

### Rank 6
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,314 impressions with a CTR of 0.30%, suggesting that improving the title or snippet could increase clicks.
- **What would make it wrong:** Recent SEO changes may not yet be reflected in the historical data.

### Rank 7
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,146 impressions with a CTR of 0.09%, showing significant room for improvement.
- **What would make it wrong:** The page may already be scheduled for optimization.

### Rank 8
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 2,442 impressions with a CTR of 0.16%, making it a high-impact optimization candidate.
- **What would make it wrong:** Low CTR may be caused by ranking position rather than the title.

### Rank 9
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,806 impressions with 0 clicks (CTR 0.00%), indicating an opportunity to improve click-through performance.
- **What would make it wrong:** The search intent may have changed since the data was collected.

### Rank 10
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,154 impressions with 0 clicks (CTR 0.00%), making it a strong CTR optimization candidate.
- **What would make it wrong:** Some impressions may come from unrelated search queries.

### Rank 11
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** High impressions with a CTR of 0.47%, indicating underperforming click-through.
- **What would make it wrong:** SERP features or rich results may reduce clicks.

### Rank 12
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,547 impressions with a CTR of 0.39%, showing an opportunity for improvement.
- **What would make it wrong:** Competitor pages may have more attractive titles.

### Rank 13
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 3,594 impressions with a CTR of 0.61%, representing a high-impact optimization opportunity.
- **What would make it wrong:** Search demand may fluctuate due to seasonality.

### Rank 14
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,924 impressions with a CTR of 0.10%, indicating low engagement despite strong visibility.
- **What would make it wrong:** The page may target broad informational queries with naturally low CTR.

### Rank 15
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,158 impressions with a CTR of 0.86%, which is still below the rule threshold.
- **What would make it wrong:** The page may already be performing well relative to its ranking position.

### Rank 16
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 2,622 impressions with a CTR of 0.23%, making it a strong optimization candidate.
- **What would make it wrong:** Changes in user search behaviour could affect CTR.

### Rank 17
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,437 impressions with a CTR of 0.21%, suggesting the page is underperforming.
- **What would make it wrong:** Poor ranking position rather than title quality may explain the low CTR.

### Rank 18
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,922 impressions with a CTR of 0.36%, indicating optimization potential.
- **What would make it wrong:** Recent content updates may not yet have influenced user behaviour.

### Rank 19
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 6,912 impressions with a CTR of 0.30%, making it the highest-impact page in the review.
- **What would make it wrong:** The page may intentionally target low-click informational searches.

### Rank 20
- **Action:** Improve Title
- **Reason Code:** CTR_FIX
- **Confidence:** High
- **Why it's there:** 1,102 impressions with a CTR of 0.18%, meeting the baseline rule for CTR optimization.
- **What would make it wrong:** External factors such as SERP features or branded searches may reduce CTR without requiring content changes.

In [15]:
top20 = ranked.head(20)

print(top20[["content_hash_id", "score", "reason_code", "action"]])

             content_hash_id  score reason_code         action
0   content_91798557b91aa5bd    100     CTR_FIX  Improve Title
1   content_8eba2d99238592aa    100     CTR_FIX  Improve Title
2   content_1380729256470720    100     CTR_FIX  Improve Title
3   content_4bcb72eff0cc03f9    100     CTR_FIX  Improve Title
4   content_c9f840183215651b    100     CTR_FIX  Improve Title
5   content_ada72ae2a2b33800    100     CTR_FIX  Improve Title
6   content_9b6b685c29cdde6b    100     CTR_FIX  Improve Title
7   content_1f3195e2491d9314    100     CTR_FIX  Improve Title
8   content_c684e826fd2cc874    100     CTR_FIX  Improve Title
9   content_91d8af19d84c05a6    100     CTR_FIX  Improve Title
10  content_06de5368fbd3bf99    100     CTR_FIX  Improve Title
11  content_3fd56bc37f6ac016    100     CTR_FIX  Improve Title
12  content_00d4fdf6e48a2d38    100     CTR_FIX  Improve Title
13  content_48cbb12a9666ea47    100     CTR_FIX  Improve Title
14  content_8ed3cff8c6597a4a    100     CTR_FIX  Improv

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [16]:
print("Leakage Check")
print("----------------------------")
print("Historical features only: PASS")
print("No future-window data used: PASS")
print("No label-derived inputs used: PASS")
print("Baseline rule is safe for evaluation.")


Leakage Check
----------------------------
Historical features only: PASS
No future-window data used: PASS
No label-derived inputs used: PASS
Baseline rule is safe for evaluation.


## Weak Picks + Leakage Check

### Weak Picks

The baseline rule identifies pages with high impressions and low CTR as candidates for title optimization. However, some recommendations may be incorrect because:

- Low CTR may be caused by poor search ranking instead of an ineffective title.
- Seasonal trends or temporary changes in search demand can reduce CTR.
- Newly published pages may not have enough historical data for a reliable decision.
- Some pages may already be scheduled for optimization, making the recommendation unnecessary.
- High impressions alone do not guarantee that changing the title will improve user engagement.

### Leakage Check

The baseline scoring rule was created using only historical Google Search Console data available at the decision time.

- Historical impressions and clicks were used to calculate CTR.
- No future-window data was included.
- No label-derived fields or product flags were used.
- The rule relies only on historical features available before making the recommendation.

**Conclusion:** No data leakage was introduced into the baseline action score, making it suitable as an honest baseline for future model comparison.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No future-window or label-derived inputs used
- [x] baseline_action_score.csv is generated
- [x] Notebook committed to work/notebooks/